# Exploratory Data Analysis

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset

import torch
from transformers import AutoTokenizer, AutoModel, pipeline
from transformers import BertConfig, BertModel
from torch.nn.functional import cosine_similarity

/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3909.81it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if yo

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

### Check correct model loaded

In [3]:
print(model.config._name_or_path)

microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract


### Download and view exaggeration dataset 

In [4]:
df = load_dataset("copenlu/scientific-exaggeration-detection")
print(df)
print(len(df["train"]))  # 100
print(len(df["test"])) 

DatasetDict({
    train: Dataset({
        features: ['original_file_id', 'press_release_conclusion', 'press_release_strength', 'abstract_conclusion', 'abstract_strength', 'exaggeration_label'],
        num_rows: 100
    })
    test: Dataset({
        features: ['original_file_id', 'press_release_conclusion', 'press_release_strength', 'abstract_conclusion', 'abstract_strength', 'exaggeration_label'],
        num_rows: 563
    })
})
100
563


In [5]:
df

DatasetDict({
    train: Dataset({
        features: ['original_file_id', 'press_release_conclusion', 'press_release_strength', 'abstract_conclusion', 'abstract_strength', 'exaggeration_label'],
        num_rows: 100
    })
    test: Dataset({
        features: ['original_file_id', 'press_release_conclusion', 'press_release_strength', 'abstract_conclusion', 'abstract_strength', 'exaggeration_label'],
        num_rows: 563
    })
})

In [6]:
# check class balance
train_df = df["train"].to_pandas()
test_df = df["test"].to_pandas()

full_df = pd.concat([train_df, test_df], ignore_index=True)
full_df["exaggeration_label"].value_counts()

exaggeration_label
same           406
exaggerates    144
downplays      113
Name: count, dtype: int64

In [7]:
#check for missing values
(full_df.isnull().mean() * 100).sort_values(ascending=False)


original_file_id            0.0
press_release_conclusion    0.0
press_release_strength      0.0
abstract_conclusion         0.0
abstract_strength           0.0
exaggeration_label          0.0
dtype: float64

### Pubmedbert configuration

In [13]:
# Initialize a BERT configuration
configuration = BertConfig()

# Initialize a model with random weights from the configuration
model = BertModel(configuration)

# Access the configuration from the model
configuration = model.config

print(model)
print(configuration)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

### Testing out Pubmedbert masked predictions
#### Beginning with example press release conclusion

In [17]:
fill_mask = pipeline(
    task="fill-mask",
    model="microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
    tokenizer="microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
    device=0,   # use -1 for CPU
)

fill_mask(
    "Nanotubes pose health [MASK], study finds.",
    top_k=5
)


Loading weights: 100%|██████████| 204/204 [00:00<00:00, 3885.84it/s, Materializing param=cls.predictions.transform.dense.weight]                 
The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler

[{'score': 0.7058877944946289,
  'token': 6131,
  'token_str': 'risks',
  'sequence': 'nanotubes pose health risks, study finds.'},
 {'score': 0.17546354234218597,
  'token': 11175,
  'token_str': 'hazards',
  'sequence': 'nanotubes pose health hazards, study finds.'},
 {'score': 0.04416218772530556,
  'token': 2209,
  'token_str': 'risk',
  'sequence': 'nanotubes pose health risk, study finds.'},
 {'score': 0.022338513284921646,
  'token': 7728,
  'token_str': 'concerns',
  'sequence': 'nanotubes pose health concerns, study finds.'},
 {'score': 0.011919156648218632,
  'token': 20694,
  'token_str': 'threats',
  'sequence': 'nanotubes pose health threats, study finds.'}]

#### Pubmedbert performs poorly on this second example. This is unsurpising since it was trained on scientific abstracts and not press releases.

In [18]:

fill_mask(
    "Nanotubes pose health risks, study [MASK].",
    top_k=5
)

[{'score': 0.35230252146720886,
  'token': 1982,
  'token_str': 'no',
  'sequence': 'nanotubes pose health risks, study no.'},
 {'score': 0.053553346544504166,
  'token': 3952,
  'token_str': 'ref',
  'sequence': 'nanotubes pose health risks, study ref.'},
 {'score': 0.03089257702231407,
  'token': 8564,
  'token_str': '##iol',
  'sequence': 'nanotubes pose health risks, studyiol.'},
 {'score': 0.016289370134472847,
  'token': 20,
  'token_str': '1',
  'sequence': 'nanotubes pose health risks, study 1.'},
 {'score': 0.015830399468541145,
  'token': 1905,
  'token_str': 'exp',
  'sequence': 'nanotubes pose health risks, study exp.'}]

#### Testing Pubmedbert with abstract masking task

In [19]:
#Find abstract examples
df["train"][0:5]["abstract_conclusion"]

['Use of the urgent referral pathway could be efficacious.',
 'Least deprived patients were 25% more likely to be initiated on anti-dementia drugs than the most deprived (adjusted incidence rate ratio 1.25, 95% confidence interval 1.19–1.31).',
 'People who reported preferences that were not met were less likely to state that treatment had helped them with their problems.',
 'Besides studies showing relief of phantom limb pain using mirrors, this is the first evidence that impeding the processes by which the brain localises a noxious stimulus can reduce pain, and that this effect reflects modulation of multimodal neural activities.',
 'Residual confounding was possible, and the analysis lacked information on folate, one of the proposed mechanisms.']

In [ ]:
#correct answer is mechanisms
fill_mask(
    "Residual confounding was possible, and the analysis lacked information on folate, one of the proposed [MASK].",
    top_k=10
)

[{'score': 0.11751561611890793,
  'token': 14536,
  'token_str': 'confounders',
  'sequence': 'residual confounding was possible, and the analysis lacked information on folate, one of the proposed confounders.'},
 {'score': 0.11009795218706131,
  'token': 9485,
  'token_str': 'mediators',
  'sequence': 'residual confounding was possible, and the analysis lacked information on folate, one of the proposed mediators.'},
 {'score': 0.10173485428094864,
  'token': 6751,
  'token_str': 'biomarkers',
  'sequence': 'residual confounding was possible, and the analysis lacked information on folate, one of the proposed biomarkers.'},
 {'score': 0.06064781919121742,
  'token': 24569,
  'token_str': 'cofactors',
  'sequence': 'residual confounding was possible, and the analysis lacked information on folate, one of the proposed cofactors.'},
 {'score': 0.060341645032167435,
  'token': 9073,
  'token_str': 'exposures',
  'sequence': 'residual confounding was possible, and the analysis lacked informat

#### The masked word (mechanims) isnt in the top 10, but many relevant biomedical terms are